In [ ]:
import pandas as pd

client_train_data = pd.read_csv("data/train/client_train.csv", low_memory=False)
invoice_train_data = pd.read_csv("data/train/invoice_train.csv",  low_memory=False)

In [ ]:
# One row per client is what we predict on, so invoices get summarised per client further down.
# First, a few per-invoice columns that those summaries are built from.
invoices = invoice_train_data.copy()
invoices["invoice_date"] = pd.to_datetime(invoices["invoice_date"])
invoices["counter_type"] = invoices["counter_type"].map({"ELEC": 0, "GAZ": 1})
invoices["diff_index"] = invoices["new_index"] - invoices["old_index"]
# All four consumption levels added up per invoice
invoices["total_cons"] = invoices[["consommation_level_1", "consommation_level_2",
                                   "consommation_level_3", "consommation_level_4"]].sum(axis=1)
by_client = invoices.groupby("client_id")

In [ ]:
import seaborn as sns
sns.countplot(data=client_train_data, x="target")
# We see a very imbalanced result

In [ ]:
import matplotlib.pyplot as plt

target_counts = client_train_data["target"].value_counts().sort_index()
plt.pie(
    target_counts,
    labels=target_counts.index,
    autopct="%1.1f%%"
)
plt.title("Client target distribution")
plt.show()

In [ ]:
# One row per client: start from the client table, add invoice summaries below
client_data = client_train_data.set_index("client_id")
y = client_data["target"]
X = client_data[["client_catg", "disrict", "region", "creation_date"]].copy()

In [ ]:
X['client_catg'] = X['client_catg'].astype(int)
X['disrict'] = X['disrict'].astype(int)

X["creation_date"] = pd.to_datetime(X["creation_date"]).astype("int64")

In [ ]:
# If the result is negative, this might indicate a fraud. This is *drumroll* FEATURE ENGINEERING!
# Per client we keep the average difference between new and old meter index
X["mean_diff_index"] = by_client["diff_index"].mean()

In [ ]:
# How many different tariff types and meters does each client have across their invoices?
# (a client switching between tariffs could be a fraud signal), plus how many invoices and what share is gas.
X["n_tarif_types"] = by_client["tarif_type"].nunique()
X["n_counters"] = by_client["counter_number"].nunique()
X["n_invoices"] = by_client.size()
X["share_gaz"] = by_client["counter_type"].mean()
X["mean_months_number"] = by_client["months_number"].mean()

In [ ]:
# Feature 1: consumption pattern per client (all four consumption levels added up per invoice).
# Fraudsters often under-report, so low, flat or zero consumption can stand out.
X["cons_mean"] = by_client["total_cons"].mean()
X["cons_std"] = by_client["total_cons"].std().fillna(0)  # clients with one invoice have no spread
X["cons_min"] = by_client["total_cons"].min()
X["cons_zero_share"] = (invoices["total_cons"] == 0).groupby(invoices["client_id"]).mean()
for level in range(1, 5):
    X[f"cons_level_{level}_mean"] = by_client[f"consommation_level_{level}"].mean()

In [ ]:
# Feature 2: negative index differences per client (the meter reading went backwards).
# Count and share, because 1 negative out of 1 invoice (100%) is not the same as 1 out of 50 (2%).
negative = invoices["diff_index"] < 0
X["n_negative_diff"] = negative.groupby(invoices["client_id"]).sum()
X["share_negative_diff"] = negative.groupby(invoices["client_id"]).mean()

In [ ]:
# Feature 3: consumption drop per client = (average of the later half of invoices - average of the earlier half)
# relative to the earlier half. A sudden drop is the classic "tampered meter" shape.
order = invoices.sort_values(["client_id", "invoice_date"])
position = order.groupby("client_id").cumcount()
n_invoices = order.groupby("client_id")["client_id"].transform("size")
is_late = position >= n_invoices / 2
early_mean = order[~is_late].groupby("client_id")["total_cons"].mean()
late_mean = order[is_late].groupby("client_id")["total_cons"].mean()
X["cons_change"] = ((late_mean - early_mean) / (early_mean + 1)).reindex(X.index).fillna(0)  # +1 avoids dividing by zero

In [ ]:
cm = X.select_dtypes(include=["number"]).corr(method="pearson")
sns.heatmap(cm)

In [ ]:
from sklearn.model_selection import train_test_split

# One row per client, so a plain split already keeps each client on one side only.
# 25% test set; stratify keeps the fraud ratio the same in both parts.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

In [ ]:
# FAST_MODE = True  -> quick sanity check: 20% of training clients, 4 random setting combos
# FAST_MODE = False -> full run: all training data, all 36 combos
FAST_MODE = True

if FAST_MODE:
    X_fit = X_train.sample(frac=0.2, random_state=42)
    y_fit = y_train.loc[X_fit.index]
else:
    X_fit, y_fit = X_train, y_train

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

# The scaler lives inside the pipeline, so the same scaling is applied to train AND test data
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lgbm", LGBMClassifier()),
])
pg = {
    "lgbm__learning_rate": [0.01, 0.1, 0.2, 1],
    "lgbm__n_estimators": [10, 100, 200],
    "lgbm__subsample_for_bin": [1000, 10000, 20000],
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

if FAST_MODE:
    gscv = RandomizedSearchCV(pipe, pg, n_iter=4, cv=cv, n_jobs=8, scoring="roc_auc", random_state=42)
else:
    gscv = GridSearchCV(pipe, pg, cv=cv, n_jobs=8, scoring="roc_auc")

gscv.fit(X_fit, y_fit)
print("Best settings:", gscv.best_params_)
print("Best cross-validated AUC:", round(gscv.best_score_, 4))

In [ ]:
y_pred = gscv.predict(X_test)

from sklearn.metrics import roc_auc_score
print("Test AUC:", round(roc_auc_score(y_test, gscv.predict_proba(X_test)[:, 1]), 4))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
